In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl"

Mounted at /content/drive
Processing ./drive/MyDrive/Master Data Science/TFM/GITHUB/modelado_multidimensional_engagement-0.1.0-py3-none-any.whl


In [2]:
import pandas as pd
import src.config as config
import os
# Entrenamiento (Noviembre)
from src.modelling.transforms import EngagementScaler
from src.modelling.similarity import KendallTauSimilarity
from src.modelling.spectral import SpectralClusteringModel

In [3]:
config.WORKPATH = "drive/MyDrive/Master Data Science/TFM/GITHUB"

In [6]:
cols = config.FEATURES_USER_BASED
workpath = config.WORKPATH
n_clusters = config.CLUSTER_USER_BASED


df_train = pd.read_parquet(os.path.join(workpath, "processed", "training_201911", "aggregated_user_based.parquet"))
df_test = pd.read_parquet(os.path.join(workpath, "processed", "validation_201912", "aggregated_user_based.parquet"))

# 1. Ranking y normalización
scaler = EngagementScaler(features=cols)
df_ranked = scaler.fit(df_train)  # Calcula means y stds
df_normalized = scaler.transform(df_ranked)
df_normalized['cluster'] = None  # placeholder

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix = similarity.fit_transform(df_normalized, features=cols)

# 3. Clustering y guardar artefactos
spectral_model = SpectralClusteringModel(n_clusters=config.CLUSTER_USER_BASED, n_init=50)
labels = spectral_model.fit(affinity_matrix, df_normalized, features=cols)

# Guardar estadísticas de entrenamiento
spectral_model.set_training_statistics(df_ranked, features=cols)

# Obtener resultados
df_clusters = spectral_model.get_clusters_dataframe(df_normalized)

# Guardar artefactos
spectral_model.save_model_artifacts(os.path.join(workpath, "models", "nov_user_based_artifacts.np"))


# ============= PROYECCIÓN (Diciembre) =============

# 1. Cargar estadísticas de entrenamiento
scaler_dec = EngagementScaler(features=cols)
scaler_dec.means = spectral_model.train_rank_means
scaler_dec.stds = spectral_model.train_rank_stds

# 2. Aplicar transformación (sin reajustar)
df_ranked_dec = df_test.copy()
df_ranked_dec = scaler_dec.rank(df_ranked_dec)
df_normalized_dec = scaler_dec.transform(df_ranked_dec)

# 3. Proyectar en espacio de centroides
df_normalized_dec["cluster"] = spectral_model.predict(df_normalized_dec[cols].values)

# 2. Similitud y afinidad
similarity = KendallTauSimilarity(method='kendall')
affinity_matrix_dec = similarity.fit_transform(df_normalized_dec, features=cols)

Computing label assignment using kmeans
Initialization complete
Iteration 0, inertia 0.04182370542776643.
Iteration 1, inertia 0.032748989926339886.
Iteration 2, inertia 0.032148954536407974.
Iteration 3, inertia 0.0317497063185399.
Iteration 4, inertia 0.03125388855461082.
Iteration 5, inertia 0.03110998734802995.
Iteration 6, inertia 0.030957252581525083.
Iteration 7, inertia 0.030835231237979842.
Iteration 8, inertia 0.030654742669048596.
Iteration 9, inertia 0.030310468647506826.
Iteration 10, inertia 0.029915385577847772.
Iteration 11, inertia 0.02970974014233593.
Iteration 12, inertia 0.02952149153949099.
Iteration 13, inertia 0.02938254369876638.
Iteration 14, inertia 0.029368244492304144.
Converged at iteration 14: strict convergence.
Initialization complete
Iteration 0, inertia 0.041644192902200484.
Iteration 1, inertia 0.03371939722825666.
Iteration 2, inertia 0.03104466681165849.
Iteration 3, inertia 0.029795267707408757.
Iteration 4, inertia 0.02896348295089637.
Iteration 5

In [8]:
affinity_matrix_dec.to_parquet(os.path.join(workpath, "processed", "validation_201912", "affinity_matrix_user_based.parquet"))
affinity_matrix.to_parquet(os.path.join(workpath, "processed", "training_201911", "affinity_matrix_user_based.parquet"))